In [1]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath("../"))

from src.geometry import generate_rectangular_loop
from src.field_solver import (
    compute_B_field,
    total_field_at_point
)

In [2]:
mm = 1e-3
T_to_mG = 1e7

In [3]:
segments = 150

coils = {}

In [4]:
x_width = 3532 * mm
x_height = 2025 * mm

coils['X1'] = generate_rectangular_loop(
    start_point=(-1700*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

coils['X2'] = generate_rectangular_loop(
    start_point=(0*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

coils['X3'] = generate_rectangular_loop(
    start_point=(1700*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

In [5]:
y_width = 3532 * mm
y_height = 1975 * mm

coils['Y1'] = generate_rectangular_loop(
    start_point=(-1766*mm, 648*mm, -979.5*mm),
    width=y_width,
    height=y_height,
    plane='xz',
    segments_per_side=segments
)

coils['Y2'] = generate_rectangular_loop(
    start_point=(-1766*mm, -1482*mm, -979.5*mm),
    width=y_width,
    height=y_height,
    plane='xz',
    segments_per_side=segments
)

In [6]:
z_width = 3582 * mm
z_height = 3582 * mm

coils['Z1'] = generate_rectangular_loop(
    start_point=(-1791*mm, 1959*mm, 750*mm),
    width=z_width,
    height=z_height,
    plane='xy',
    segments_per_side=segments
)

coils['Z2'] = generate_rectangular_loop(
    start_point=(-1791*mm, 1959*mm, -750*mm),
    width=z_width,
    height=z_height,
    plane='xy',
    segments_per_side=segments
)

In [7]:
PMTs = {
    'PMT1': np.array([-760, -834, 0]) * mm,
    'PMT2': np.array([-760, 0, 0]) * mm,
    'PMT3': np.array([-530, 1173, 0]) * mm,
    'PMT4': np.array([530, -837, 0]) * mm,
    'PMT5': np.array([530, 3, 0]) * mm,
    'PMT6': np.array([530, 1173, 0]) * mm,
}

In [8]:
coil_names = ['X1', 'X2', 'X3', 'Y1', 'Y2', 'Z1', 'Z2']

In [9]:
n_pmts = len(PMTs)
n_coils = len(coil_names)

Ax = np.zeros((n_pmts, n_coils))
Ay = np.zeros((n_pmts, n_coils))
Az = np.zeros((n_pmts, n_coils))

In [10]:
for j, coil in enumerate(coil_names):

    # Set only ONE coil to 1A
    currents = {name:0.0 for name in coil_names}
    currents[coil] = 1.0

    for i, (pmt_name, position) in enumerate(PMTs.items()):

        B = total_field_at_point(position, coils, currents)

        # Convert to mG immediately
        B *= T_to_mG

        Ax[i, j] = B[0]
        Ay[i, j] = B[1]
        Az[i, j] = B[2]

In [11]:
print("Ax matrix:")
print(Ax)

print("\nAy matrix:")
print(Ay)

print("\nAz matrix:")
print(Az)

Ax matrix:
[[ 2.34247525  2.92621458  0.47600345  0.33560675 -0.50856593  0.55555116
  -0.55555116]
 [ 2.46777398  2.94668618  0.55838691  0.50856593 -0.33560675  0.63137798
  -0.63137798]
 [ 1.76598033  3.83298989  0.58129597 -0.2717508  -0.08109841  0.33939176
  -0.33939176]
 [ 0.58129597  3.83298989  1.76598033 -0.22716585  0.2968306  -0.33939176
   0.33939176]
 [ 0.68486221  3.6104027   1.94856656 -0.2968306   0.22716585 -0.38860732
   0.38860732]
 [ 0.58129597  3.83298989  1.76598033  0.2717508   0.08109841 -0.33939176
   0.33939176]]

Ay matrix:
[[-0.74969932  0.83486116  0.17662924  1.30618655  3.35779258  0.92789403
  -0.92789403]
 [-0.08895734  0.08689127  0.03186184  3.35779258  1.30618655  0.1024987
  -0.1024987 ]
 [ 0.6228299  -0.86389752 -0.22062588  3.7226229   0.44643627 -0.96519517
   0.96519517]
 [-0.22062588 -0.86389752  0.6228299   1.35413122  3.3337897   0.96519517
  -0.96519517]
 [-0.03839645 -0.07255073  0.08279248  3.3337897   1.35413122  0.10487018
  -0.10487018

In [12]:
A = np.vstack([Ax, Ay, Az])

print("Full response matrix shape:")
print(A.shape)

Full response matrix shape:
(18, 7)


In [13]:
print("Rows 0-5   -> Bx at PMTs")
print("Rows 6-11  -> By at PMTs")
print("Rows 12-17 -> Bz at PMTs")

Rows 0-5   -> Bx at PMTs
Rows 6-11  -> By at PMTs
Rows 12-17 -> Bz at PMTs


In [14]:
#example
I_test = np.array([
    10,
    5,
    10,
    -3,
    -3,
    7,
    7
])

In [15]:
B_pred = A @ I_test

print(B_pred)

[ 43.33473744  44.47616227  43.69626019  42.4287183   44.59529547
  41.5791649  -15.54833242 -14.12843607 -12.80462488 -14.36121013
 -13.98255609 -12.20973013  37.49136754  36.91267497  37.29485808
  37.2469548   36.61235912  37.27694795]


In [16]:
Bx_pred = B_pred[0:6]
By_pred = B_pred[6:12]
Bz_pred = B_pred[12:18]

for i, pmt in enumerate(PMTs.keys()):

    mag = np.sqrt(
        Bx_pred[i]**2 +
        By_pred[i]**2 +
        Bz_pred[i]**2
    )

    print(f"\n{pmt}")
    print("-"*30)

    print(f"Bx = {Bx_pred[i]:.3f} mG")
    print(f"By = {By_pred[i]:.3f} mG")
    print(f"Bz = {Bz_pred[i]:.3f} mG")
    print(f"|B| = {mag:.3f} mG")


PMT1
------------------------------
Bx = 43.335 mG
By = -15.548 mG
Bz = 37.491 mG
|B| = 59.374 mG

PMT2
------------------------------
Bx = 44.476 mG
By = -14.128 mG
Bz = 36.913 mG
|B| = 59.500 mG

PMT3
------------------------------
Bx = 43.696 mG
By = -12.805 mG
Bz = 37.295 mG
|B| = 58.858 mG

PMT4
------------------------------
Bx = 42.429 mG
By = -14.361 mG
Bz = 37.247 mG
|B| = 58.256 mG

PMT5
------------------------------
Bx = 44.595 mG
By = -13.983 mG
Bz = 36.612 mG
|B| = 59.369 mG

PMT6
------------------------------
Bx = 41.579 mG
By = -12.210 mG
Bz = 37.277 mG
|B| = 57.162 mG


In [17]:
import numpy as np
import os

# Create data folder if not existing
os.makedirs("../data", exist_ok=True)

# Save matrices
np.save("../data/Ax.npy", Ax)
np.save("../data/Ay.npy", Ay)
np.save("../data/Az.npy", Az)
np.save("../data/A.npy", A)

print("Matrices saved successfully.")

Matrices saved successfully.
